# Preprocessing — UNSW-NB15 Intrusion Detection Dataset

This notebook turns the raw UNSW-NB15 training and testing sets into clean, fully numeric tables that a machine-learning model can consume. Every transformation (encoding, scaling) is **fit on the training set only** and then applied to the testing set, so that no information from the test set leaks into how we prepare the data — this mirrors how a model would be deployed in practice, where future traffic is unseen at training time.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

DATA_DIR = Path.cwd().parent / "data"
TRAIN_PATH = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH = DATA_DIR / "UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training set:", train_df.shape)
print("Testing set:", test_df.shape)

Training set: (82332, 45)
Testing set: (175341, 45)


## Step 1 — Handling missing values

**Decision:** Numeric columns with missing values are filled with the **median** of that column from the training set (medians are robust to the extreme outliers common in network traffic, e.g. a handful of very long-duration sessions). Categorical columns with missing values are filled with the literal string `"missing"`, so a gap in a sensor's readings is preserved as its own signal rather than silently guessed at. If a real-world data feed drops a field, that absence can itself be meaningful (e.g. a firewall failing to log a service), so we don't want to erase that information by imputing it away.

In [2]:
print("Missing values in training set:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print("\nMissing values in testing set:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

Missing values in training set:
Series([], dtype: int64)

Missing values in testing set:
Series([], dtype: int64)


In [3]:
categorical_cols = ["proto", "service", "state", "attack_cat"]
numeric_cols = [
    c for c in train_df.columns
    if c not in categorical_cols + ["id", "label"]
]

# Fill numeric NaNs with the training-set median (computed once, reused for test).
medians = train_df[numeric_cols].median()
train_df[numeric_cols] = train_df[numeric_cols].fillna(medians)
test_df[numeric_cols] = test_df[numeric_cols].fillna(medians)

# Fill categorical NaNs with an explicit "missing" marker.
for col in categorical_cols:
    train_df[col] = train_df[col].fillna("missing")
    test_df[col] = test_df[col].fillna("missing")

print("Remaining nulls in train:", train_df.isnull().sum().sum())
print("Remaining nulls in test:", test_df.isnull().sum().sum())

Remaining nulls in train: 0
Remaining nulls in test: 0


## Step 2 — Encoding categorical columns

We have four categorical columns, and we don't treat them all the same way because they differ in how many distinct values they hold:

- **`proto` (131 distinct protocols):** encoded with **label encoding**. One-hot encoding this column would add 131 new columns for a single field, most of them almost always zero — that bloats the feature space and slows training for little benefit. Label encoding keeps it to one column. The trade-off is that label encoding implies an arbitrary numeric order between protocols (e.g. `tcp=2 < udp=5`) which tree-based models (e.g. random forests, gradient boosting) can handle fine since they split on thresholds, but linear/distance-based models could misread as a real ranking — worth remembering if a linear model is used downstream.
- **`service` (13 distinct values) and `state` (7 distinct values):** encoded with **one-hot encoding**. Low cardinality keeps the extra columns manageable, and these are unordered categories (there's no natural ranking between "http" and "ftp", or between connection states) — one-hot avoids implying a false order that label encoding would introduce.
- **`attack_cat` (the specific attack family, e.g. "DoS", "Exploits", "Normal"):** encoded with **label encoding** purely for reference/analysis. It is **not included as a predictive feature** for the binary `label` target, because it is derived from the exact same ground truth as `label` (if you know the attack category, you trivially know whether it's an attack) — including it would leak the answer into the model and produce artificially perfect results that wouldn't hold on real traffic.

In [4]:
# --- proto: label encoding (high cardinality) ---
proto_encoder = LabelEncoder()
proto_encoder.fit(train_df["proto"])

def safe_label_transform(encoder, series):
    """Map unseen categories in test data to a new 'unknown' bucket instead of erroring."""
    known = set(encoder.classes_)
    mapped = series.where(series.isin(known), other="__unknown__")
    if "__unknown__" not in encoder.classes_:
        encoder.classes_ = np.append(encoder.classes_, "__unknown__")
    return encoder.transform(mapped)

train_df["proto_encoded"] = proto_encoder.transform(train_df["proto"])
test_df["proto_encoded"] = safe_label_transform(proto_encoder, test_df["proto"])

# --- attack_cat: label encoding, kept as a reference column only (not a feature) ---
attack_cat_encoder = LabelEncoder()
attack_cat_encoder.fit(train_df["attack_cat"])
train_df["attack_cat_encoded"] = attack_cat_encoder.transform(train_df["attack_cat"])
test_df["attack_cat_encoded"] = safe_label_transform(attack_cat_encoder, test_df["attack_cat"])

# --- service, state: one-hot encoding (low cardinality, unordered) ---
onehot_cols = ["service", "state"]
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehot_encoder.fit(train_df[onehot_cols])

onehot_train = pd.DataFrame(
    onehot_encoder.transform(train_df[onehot_cols]),
    columns=onehot_encoder.get_feature_names_out(onehot_cols),
    index=train_df.index,
)
onehot_test = pd.DataFrame(
    onehot_encoder.transform(test_df[onehot_cols]),
    columns=onehot_encoder.get_feature_names_out(onehot_cols),
    index=test_df.index,
)

train_df = pd.concat([train_df, onehot_train], axis=1)
test_df = pd.concat([test_df, onehot_test], axis=1)

print("New one-hot columns:", list(onehot_train.columns))

New one-hot columns: ['service_-', 'service_dhcp', 'service_dns', 'service_ftp', 'service_ftp-data', 'service_http', 'service_irc', 'service_pop3', 'service_radius', 'service_smtp', 'service_snmp', 'service_ssh', 'service_ssl', 'state_ACC', 'state_CLO', 'state_CON', 'state_FIN', 'state_INT', 'state_REQ', 'state_RST']


## Step 3 — Scaling numeric features

**Decision:** Scale all numeric traffic features (packet counts, byte counts, timing, TTL, jitter, etc.) with **`StandardScaler`**, which rescales each column to have mean 0 and standard deviation 1. Network features in this dataset span wildly different ranges (e.g. `dur` is fractions of a second while `sbytes` can be in the millions), and many algorithms — especially distance-based or gradient-based ones — treat larger raw numbers as "more important" unless everything is put on a common scale. As with the encoders, the scaler is **fit on the training set only** and reused on the test set, so the test set's own distribution can't influence how we transform it.

In [5]:
scale_cols = numeric_cols + ["proto_encoded"]

scaler = StandardScaler()
scaler.fit(train_df[scale_cols])

train_df[scale_cols] = scaler.transform(train_df[scale_cols])
test_df[scale_cols] = scaler.transform(test_df[scale_cols])

train_df[scale_cols].describe().T[["mean", "std"]].head()

,mean,std
dur,-5.523337e-18,1.000006
spkts,8.285005e-18,1.000006
dpkts,0.000000e+00,1.000006
sbytes,-2.416460e-18,1.000006
dbytes,1.657001e-17,1.000006


## Step 4 — Assembling the final feature set

We drop the original text categorical columns (`proto`, `service`, `state`, `attack_cat`) now that their encoded versions exist, and drop the `id` column since it's just a row identifier with no predictive meaning. `attack_cat_encoded` is kept in the saved file for reference/analysis but should be excluded from the feature matrix `X` when training a binary classifier, since (as explained above) it leaks the target. The binary `label` column (0 = normal, 1 = attack) remains as the target for modeling.

In [6]:
drop_cols = ["id", "proto", "service", "state", "attack_cat"]

train_clean = train_df.drop(columns=drop_cols)
test_clean = test_df.drop(columns=drop_cols)

# Put label at the end for readability; attack_cat_encoded kept as reference only.
cols = [c for c in train_clean.columns if c not in ("label", "attack_cat_encoded")] + ["attack_cat_encoded", "label"]
train_clean = train_clean[cols]
test_clean = test_clean[cols]

print("Final train shape:", train_clean.shape)
print("Final test shape:", test_clean.shape)
train_clean.head()

Final train shape: (82332, 62)
Final test shape: (175341, 62)


,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,service_ssl,state_ACC,state_CLO,state_CON,state_FIN,state_INT,state_REQ,state_RST,attack_cat_encoded,label
0,-0.213727,-0.124455,-0.151816,-0.043684,-0.087369,0.057181,0.71944,-0.820395,0.643913,-0.263498,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6,0
1,-0.213728,-0.124455,-0.151816,-0.036308,-0.087369,0.286565,0.71944,-0.820395,4.539351,-0.263498,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6,0
2,-0.213729,-0.124455,-0.151816,-0.040351,-0.087369,0.791209,0.71944,-0.820395,4.391459,-0.263498,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6,0
3,-0.213729,-0.124455,-0.151816,-0.041330,-0.087369,0.566923,0.71944,-0.820395,2.977031,-0.263498,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6,0
4,-0.213728,-0.124455,-0.151816,-0.034187,-0.087369,0.118350,0.71944,-0.820395,4.369219,-0.263498,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6,0


## Step 5 — Saving the cleaned datasets

The cleaned, fully numeric train and test sets are written to `data/train_clean.csv` and `data/test_clean.csv`. These are now ready to be split into `X` (features) and `y` (the `label` column) for model training, with `attack_cat_encoded` excluded from `X`.

In [7]:
train_clean.to_csv(DATA_DIR / "train_clean.csv", index=False)
test_clean.to_csv(DATA_DIR / "test_clean.csv", index=False)

print("Saved:", DATA_DIR / "train_clean.csv")
print("Saved:", DATA_DIR / "test_clean.csv")

Saved: D:\IS\ids-ml-project\data\train_clean.csv
Saved: D:\IS\ids-ml-project\data\test_clean.csv


## Step 6 — Saving preprocessing artifacts for reuse

The fitted encoders and scaler above are saved to `models/preprocessing.joblib`, alongside the final feature column order and some reference statistics (per-column min/median/max, and the categorical options seen in training). This lets any other tool — such as the Streamlit dashboard — apply the **exact same** preprocessing to brand-new, raw traffic records before feeding them to a saved model, instead of re-deriving encoders from scratch (which could silently drift from what the model was trained on).

In [8]:
import joblib

MODELS_DIR = Path.cwd().parent / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Reload a fresh, untouched copy of the raw training data to compute reference
# stats (train_df above has already been encoded/scaled in place).
raw_train_df = pd.read_csv(TRAIN_PATH)
raw_train_df[numeric_cols] = raw_train_df[numeric_cols].fillna(medians)

raw_stats = raw_train_df[numeric_cols].agg(["min", "median", "max"]).T
raw_stats_dict = raw_stats.to_dict(orient="index")

categorical_options = {
    "proto": sorted(raw_train_df["proto"].dropna().unique().tolist()),
    "service": sorted(raw_train_df["service"].dropna().unique().tolist()),
    "state": sorted(raw_train_df["state"].dropna().unique().tolist()),
}

feature_columns = [c for c in train_clean.columns if c not in ("label", "attack_cat_encoded")]

artifacts = {
    "proto_encoder": proto_encoder,
    "attack_cat_encoder": attack_cat_encoder,
    "onehot_encoder": onehot_encoder,
    "onehot_cols": onehot_cols,
    "numeric_cols": numeric_cols,
    "scale_cols": scale_cols,
    "medians": medians,
    "scaler": scaler,
    "categorical_options": categorical_options,
    "raw_stats": raw_stats_dict,
    "feature_columns": feature_columns,
}

artifacts_path = MODELS_DIR / "preprocessing.joblib"
joblib.dump(artifacts, artifacts_path)
print("Saved:", artifacts_path)

Saved: D:\IS\ids-ml-project\models\preprocessing.joblib


## Summary of preprocessing decisions

| Step | Decision | Why |
|---|---|---|
| Missing numeric values | Filled with training-set median | Robust to outliers, avoids leaking test statistics |
| Missing categorical values | Filled with `"missing"` marker | Preserves the fact that data was absent rather than guessing |
| `proto` (131 categories) | Label encoding | One-hot would add 131 sparse columns; label encoding is compact and works well with tree-based models |
| `service`, `state` (low cardinality) | One-hot encoding | Small number of unordered categories; avoids implying a false ranking |
| `attack_cat` | Label encoding, kept as reference only | Directly derived from the target; using it as a feature would leak the answer |
| Numeric features | `StandardScaler` (fit on train, applied to test) | Puts wildly different feature ranges on a common scale for scale-sensitive algorithms |
| `label` | Left untouched | This is the binary target (0 = normal, 1 = attack) that any model will be trained to predict |